# **🗺️ Mapatón Regional - Perú 2026**

*Notebook desarrollado por Evelyn Cabrera Arias.*

Este notebook contiene el pipeline de procesamiento espacial y visualización interactiva de edificaciones mapeadas por voluntarios en Perú.
El objetivo principal es visibilizar el esfuerzo de mapeo colaborativo y analizar la concentración y exposición espacial de las estructuras frente al riesgo climático.

📋 Estructura del Proceso
1. **Carga y Descarga de Datos:** Obtención automática de los datos de edificaciones a partir de HDX (Humanitarian Data Exchange).
2. **Procesamiento Métrico:** Reproyección al sistema métrico local (UTM 18S) para el cálculo exacto de áreas ($\text{m}^2$) y centroides geográficos reales.
3. **Modelamiento de Exposición Espacial:** Construcción de una rejilla (*Grid Binning*) que agrupa los datos para calcular la concentración de edificaciones y un índice de exposición basado en el tamaño de la huella constructiva y la densidad local.
4. **Generación del Dashboard Interactivo:** Creación de un mapa interactivo con controles de capas dinámicos (Leaflet) para alternar entre Concentración de Edificaciones e Índice de Exposición Relativa.
---



In [6]:
import geopandas as gpd
import folium
import requests
import json
import numpy as np

# =========================================================================
# PARÁMETROS GLOBALES DEL PROYECTO
# =========================================================================
LOCAL_BUILDINGS_FILE = "Peru_buildings.geojson"
BUILDINGS_URL = "https://data.humdata.org/dataset/11e31986-1763-47ba-993c-7e45303441f6/resource/55513fec-beb5-4cf5-a25e-21e16708dba9/download/peru.geojson"
CARTO_API_KEY = "MI_API_KEY"  # Reemplazar o configurar restricciones por dominio en CARTO
GRID_RESOLUTION = 0.05  # Tamaño del binning espacial (aprox. 5.5 km)

# =========================================================================
# 1. CARGA Y DESCARGA DE DATOS DE EDIFICACIONES
# =========================================================================
print("1. Cargando el archivo GeoJSON de edificaciones...")
try:
    gdf = gpd.read_file(LOCAL_BUILDINGS_FILE)
except Exception as e:
    print(f"Descargando archivo desde HDX... ({e})")
    response = requests.get(BUILDINGS_URL, stream=True)
    if response.status_code == 200:
        with open(LOCAL_BUILDINGS_FILE, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    gdf = gpd.read_file(LOCAL_BUILDINGS_FILE)

total_edificios = len(gdf)
print(f"📊 CARGA EXITOSA: {total_edificios:,} edificaciones listas para procesar.")

# =========================================================================
# 2. PROCESAMIENTO GEOGRÁFICO Y MÉTRICO
# =========================================================================
print("2. Reproyectando a sistema métrico (UTM 18S) para calcular áreas...")
gdf_metric = gdf.to_crs(epsg=32718) # EPSG:32718 (UTM Zona 18S para Perú)
gdf['area_m2'] = gdf_metric.geometry.area

# Métricas globales para KPIs
area_promedio_global = gdf['area_m2'].mean()
porcentaje_pequeños_global = (len(gdf[gdf['area_m2'] < 10]) / total_edificios) * 100

# Modelamiento analítico preliminar basado en tamaño de huella y densidad edilicia
gdf['score_tamano'] = gdf['area_m2'].apply(lambda x: 80.0 if x < 10 else (45.0 if x < 30 else 15.0))

centroids_geographic = gdf_metric.geometry.centroid.to_crs(epsg=4326)
gdf['lat'] = centroids_geographic.y
gdf['lon'] = centroids_geographic.x

# =========================================================================
# 3. REJILLA ESPACIAL (GRID BINNING) Y MODELO DE EXPOSICIÓN
# =========================================================================
print("3. Construyendo rejilla de agregación espacial...")
gdf['lat_bin'] = (gdf['lat'] / GRID_RESOLUTION).round() * GRID_RESOLUTION
gdf['lon_bin'] = (gdf['lon'] / GRID_RESOLUTION).round() * GRID_RESOLUTION

grid_df = gdf.groupby(['lat_bin', 'lon_bin']).agg(
    edificaciones=('lat', 'count'),
    score_promedio=('score_tamano', 'mean')
).reset_index()

# Normalización del factor de densidad edilicia relativa
max_edif = grid_df['edificaciones'].max()
grid_df['densidad_factor'] = (grid_df['edificaciones'] / max_edif) * 100

# Índice analítico combinado (Ajustado a la distribución real de los datos)
grid_df['indice_exposicion'] = (grid_df['score_promedio'] * 0.5) + (grid_df['densidad_factor'] * 0.5)
grid_df['indice_exposicion'] = grid_df['indice_exposicion'].clip(5.0, 85.0)

suma_verificacion = grid_df['edificaciones'].sum()
print(f"✔️ Validación de integridad: {suma_verificacion:,} edificaciones agrupadas correctamente.")

raw_data = grid_df[['lat_bin', 'lon_bin', 'edificaciones', 'indice_exposicion']].values.tolist()
raw_json = json.dumps(raw_data, separators=(',', ':'))

# =========================================================================
# 4. CONFIGURACIÓN DEL MAPA BASE (FOLIUM / CARTO VOYAGER)
# =========================================================================
print("4. Configurando mapa interactivo con Carto Voyager...")
m = folium.Map(
    location=[-9.19, -75.0152],
    zoom_start=6,
    tiles=None
)

# Mapeo base CARTO con etiquetas forzadas en español
carto_tile_url = f"https://{{s}}.basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}.png?key={CARTO_API_KEY}&lang=es"
folium.TileLayer(
    tiles=carto_tile_url,
    name='Mapa Base (Carto Voyager)',
    attr='&copy; OpenStreetMap contributors &copy; CARTO',
    subdomains='abcd',
    max_zoom=20
).add_to(m)

# Capa de ríos optimizada y filtrada para la extensión territorial de Perú
print("5. Añadiendo capa de hidrografía recortada...")
try:
    rivers_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_rivers_lake_centerlines.geojson"
    rivers_gdf = gpd.read_file(rivers_url)
    # Filtro espacial estricto para recortar solo a la caja de Perú y evitar cargar datos globales pesados
    peru_rivers = rivers_gdf.cx[-82.0:-68.5, -18.5:0.5]

    folium.GeoJson(
        peru_rivers,
        name='Ríos Principales',
        style_function=lambda x: {'color': '#2171b5', 'weight': 1.6, 'opacity': 0.6},
        tooltip=folium.GeoJsonTooltip(fields=['name'], aliases=['Río:'], localize=True, sticky=True)
    ).add_to(m)
except Exception as e:
    print("Nota sobre ríos:", e)

# =========================================================================
# 5. GENERACIÓN DEL DASHBOARD INTERACTIVO (HTML / CSS / JS)
# =========================================================================
print("6. Integrando interfaz flotante y controles analíticos...")

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<style>
    .top-left-card {{
        position: absolute; top: 20px; left: 20px; z-index: 100000;
        background: rgba(255, 255, 255, 0.96); border-radius: 10px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2); padding: 14px 18px;
        font-family: 'Calibri', Arial, sans-serif; color: #1F3B57; width: 360px; pointer-events: auto;
    }}
    .top-left-card h3 {{ margin: 0 0 4px 0; font-size: 16px; font-weight: bold; }}
    .top-left-card p {{ margin: 0 0 10px 0; font-size: 11px; color: #555; }}
    .kpi-row {{ display: flex; justify-content: space-between; background: #F7F4EE; border-radius: 8px; padding: 8px 10px; text-align: center; }}
    .kpi-item .val {{ font-size: 14px; font-weight: bold; color: #D0021B; }}
    .kpi-item .lbl {{ font-size: 9px; color: #666; text-transform: uppercase; }}

    .top-right-card {{
        position: absolute; top: 20px; right: 20px; z-index: 100000;
        background: rgba(255, 255, 255, 0.96); border-radius: 10px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2); padding: 12px 16px;
        font-family: 'Calibri', Arial, sans-serif; color: #1F3B57; width: 250px; pointer-events: auto;
    }}
    .top-right-card h4 {{ margin: 0 0 8px 0; font-size: 13px; border-bottom: 1px solid #ddd; padding-bottom: 4px; }}
    .radio-option {{ font-size: 12px; margin-bottom: 6px; cursor: pointer; display: flex; align-items: center; }}
    .radio-option input {{ margin-right: 8px; }}

    .bottom-left-legend {{
        position: absolute; bottom: 30px; left: 20px; z-index: 100000;
        background: rgba(255, 255, 255, 0.96); border-radius: 10px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2); padding: 10px 14px;
        font-family: 'Calibri', Arial, sans-serif; color: #1F3B57; font-size: 11px; width: 180px; pointer-events: auto;
    }}
    .legend-title {{ font-weight: bold; margin-bottom: 4px; }}
    .legend-scale {{ display: flex; flex-direction: column; gap: 3px; }}
    .legend-item {{ display: flex; align-items: center; gap: 8px; }}
    .color-box {{ width: 14px; height: 12px; border-radius: 2px; }}

    @media(max-width: 768px) {{
        .top-left-card {{ width: 90%; left: 5%; top: 10px; }}
        .top-right-card {{ right: 5%; top: 130px; }}
        .bottom-left-legend {{ display: none; }}
    }}
</style>
</head>
<body>
    <div class="top-left-card">
        <h3>Macro Mapatón Regional • Perú</h3>
        <p>Edificaciones mapeadas por voluntarios para visibilizar comunidades y apoyar la prevención ante riesgos climáticos.</p>
        <p style="margin: 0 0 8px 0; font-size: 10px; color: #D0021B; font-weight: bold;">
            Desarrollado por: Evelyn Cabrera Arias
        </p>
        <div class="kpi-row">
            <div class="kpi-item">
                <div class="val">{total_edificios:,}</div>
                <div class="lbl">Edificaciones</div>
            </div>
            <div class="kpi-item">
                <div class="val">{porcentaje_pequeños_global:.1f}%</div>
                <div class="lbl">&lt; 10 m² (Huellas Pequeñas)</div>
            </div>
            <div class="kpi-item">
                <div class="val">{area_promedio_global:.1f} m²</div>
                <div class="lbl">Área Promedio</div>
            </div>
        </div>
    </div>

    <div class="top-right-card">
        <h4>Ver capa</h4>
        <div class="radio-option">
            <input type="radio" name="mapLayer" value="concentracion" checked onchange="changeLayer(this.value)"> Concentración de Edificaciones
        </div>
        <div class="radio-option">
            <input type="radio" name="mapLayer" value="vulnerabilidad" onchange="changeLayer(this.value)"> Índice de Exposición Relativa
        </div>
    </div>

    <div class="bottom-left-legend" id="legendBox">
        <div class="legend-title" id="legendTitle">Edificaciones por celda</div>
        <div style="font-size: 10px; color: #666; margin-bottom: 6px;">celda ≈ 5.5 km</div>
        <div class="legend-scale" id="legendScaleContent">
            <div class="legend-item"><div class="color-box" style="background:#feedde;"></div> 1 – 9</div>
            <div class="legend-item"><div class="color-box" style="background:#fdbe85;"></div> 10 – 49</div>
            <div class="legend-item"><div class="color-box" style="background:#fd8d3c;"></div> 50 – 199</div>
            <div class="legend-item"><div class="color-box" style="background:#e6550d;"></div> 200 – 999</div>
            <div class="legend-item"><div class="color-box" style="background:#a63603;"></div> 1000 – 2999</div>
            <div class="legend-item"><div class="color-box" style="background:#7f2704;"></div> 3000 o más</div>
        </div>
    </div>

    <script>
        window.addEventListener('load', function() {{
            const map = {m.get_name()};
            const rawGridData = {raw_json};
            let dynamicLayer = L.layerGroup();

            function getColor(val, metric) {{
                if (metric === 'concentracion') {{
                    return val > 3000 ? '#7f2704' :
                           val > 1000 ? '#a63603' :
                           val > 199  ? '#e6550d' :
                           val > 49   ? '#fd8d3c' :
                           val > 9    ? '#fdbe85' : '#feedde';
                }} else {{
                    return val > 50 ? '#67000d' :
                           val > 40 ? '#a50f15' :
                           val > 25 ? '#de2d26' :
                           val > 15 ? '#fb6a4a' : '#fcae91';
                }}
            }}

            function renderMap(metric) {{
                dynamicLayer.clearLayers();

                rawGridData.forEach(row => {{
                    let lat = row[0];
                    let lon = row[1];
                    let edif = row[2];
                    let exp = row[3];

                    let valForColor = (metric === 'concentracion') ? edif : exp;
                    let fillColor = getColor(valForColor, metric);

                    let bounds = [[lat - 0.025, lon - 0.025], [lat + 0.025, lon + 0.025]];
                    let rect = L.rectangle(bounds, {{
                        color: fillColor,
                        weight: 0.5,
                        fillColor: fillColor,
                        fillOpacity: 0.75
                    }});

                    let tooltipContent = (metric === 'concentracion')
                        ? `<b>${{edif.toLocaleString()}}</b> edificaciones (celda ≈ 5.5 km)<br>Exposición Relativa: ${{exp.toFixed(1)}}%`
                        : `<b>Exposición Relativa: ${{exp.toFixed(1)}}%</b><br>Edificaciones en celda: ${{edif.toLocaleString()}}`;

                    rect.bindTooltip(tooltipContent, {{ sticky: true, direction: 'top' }});
                    dynamicLayer.addLayer(rect);
                }});

                dynamicLayer.addTo(map);
            }}

            window.changeLayer = function(metric) {{
                renderMap(metric);
                let legendTitle = document.getElementById('legendTitle');
                let legendScale = document.getElementById('legendScaleContent');

                if (metric === 'concentracion') {{
                    legendTitle.innerText = "Edificaciones por celda";
                    legendScale.innerHTML = `
                        <div class="legend-item"><div class="color-box" style="background:#feedde;"></div> 1 – 9</div>
                        <div class="legend-item"><div class="color-box" style="background:#fdbe85;"></div> 10 – 49</div>
                        <div class="legend-item"><div class="color-box" style="background:#fd8d3c;"></div> 50 – 199</div>
                        <div class="legend-item"><div class="color-box" style="background:#e6550d;"></div> 200 – 999</div>
                        <div class="legend-item"><div class="color-box" style="background:#a63603;"></div> 1000 – 2999</div>
                        <div class="legend-item"><div class="color-box" style="background:#7f2704;"></div> 3000 o más`;
                }} else {{
                    legendTitle.innerText = "Índice Exposición Relativa";
                    legendScale.innerHTML = `
                        <div class="legend-item"><div class="color-box" style="background:#fcae91;"></div> 5% – 15%</div>
                        <div class="legend-item"><div class="color-box" style="background:#fb6a4a;"></div> 15% – 25%</div>
                        <div class="legend-item"><div class="color-box" style="background:#de2d26;"></div> 25% – 40%</div>
                        <div class="legend-item"><div class="color-box" style="background:#a50f15;"></div> 40% – 50%</div>
                        <div class="legend-item"><div class="color-box" style="background:#67000d;"></div> > 50%</div>
                    `;
                }}
            }};

            renderMap('concentracion');
        }});
    </script>
</body>
</html>
"""

m.get_root().html.add_child(folium.Element(dashboard_html))
folium.LayerControl(collapsed=True).add_to(m)

output_file = "mapa_mapaton_peru_2026.html"
m.save(output_file)
print(f"\n¡Proceso finalizado con éxito! Archivo guardado como: '{output_file}'")

1. Cargando el archivo GeoJSON de edificaciones...
📊 CARGA EXITOSA: 475,473 edificaciones listas para procesar.
2. Reproyectando a sistema métrico (UTM 18S) para calcular áreas...
3. Construyendo rejilla de agregación espacial...
✔️ Validación de integridad: 475,473 edificaciones agrupadas correctamente.
4. Configurando mapa interactivo con Carto Voyager...
5. Añadiendo capa de hidrografía recortada...
6. Integrando interfaz flotante y controles analíticos...

¡Proceso finalizado con éxito! Archivo guardado como: 'mapa_mapaton_peru_2026.html'
